# Twinkle 客户端 - 多模态 LoRA 训练（LaTeX OCR）

> 🎯 **训练目标**：通过多模态 LoRA 微调，让模型学会 **LaTeX OCR** —— 从图片中识别并输出 LaTeX 数学公式。

本 Notebook 演示如何通过 **Twinkle 客户端** 使用 LoRA 微调一个多模态语言模型，实现 **LaTeX OCR**（从图片识别 LaTeX 公式）。

### 与文本训练的区别

~

相比 `self_cognition.ipynb`（纯文本训练），多模态训练的主要差异：
- 数据集包含 **图片**，消息中用 `<image>` 标记图片位置
- 使用 `Qwen3_5Template`（支持视觉输入）而非通用 `Template`
- 使用 `LazyDataset`（延迟加载）而非 `Dataset`，避免图片数据一次性加载到内存
- 训练循环中需要将 numpy/torch 张量转为列表，适配网络传输格式

### 整体流程

```
初始化客户端 -> 定义数据预处理器 -> 准备数据集 -> 配置模型 -> 训练循环 -> 保存检查点
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 服务端已启动 | 需支持多模态模型（如 Qwen3.6 系列） |
| 环境变量 | `.env` 文件或环境中设置 `MODELSCOPE_TOKEN` |
| 依赖安装 | `pip install twinkle-kit[tinker]` |

> 💡 **获取 Token**：访问 [ModelScope Token 页面](https://www.modelscope.cn/my/access/token) 获取你的 `MODELSCOPE_TOKEN`，并设置为环境变量：`export MODELSCOPE_TOKEN=<你的Token>`


## 🚀 零卡训练服务化（Serverless Training）

本 Notebook 运行在 **ModelScope 零卡训练平台** 上。你无需自备 GPU，只需在 Notebook 中编写训练逻辑，平台会自动调度云端 GPU 资源完成训练。

### 架构示意图

```
┌─────────────────────────────────────────────────────────────┐
│                    你的 Notebook（CPU 环境）                   │
│                                                             │
│   ┌──────────┐    HTTP / gRPC     ┌──────────────────────┐  │
│   │  Twinkle │ ─────────────────► │   ModelScope 云端     │  │
│   │  Client  │ ◄───────────────── │   GPU 训练集群        │  │
│   └──────────┘    训练结果返回      │                      │  │
│       │                           │  ┌────┐ ┌────┐ ┌────┐│  │
│       │  构造数据                   │  │GPU0│ │GPU1│ │... ││  │
│       │  发送训练请求               │  └────┘ └────┘ └────┘│  │
│       │  接收指标/检查点            │  模型加载 + LoRA 训练  │  │
│       ▼                           └──────────────────────┘  │
│   ┌──────────┐                                              │
│   │ 数据准备  │  Dataset / DataLoader / Preprocessor         │
│   └──────────┘                                              │
└─────────────────────────────────────────────────────────────┘
```

### 核心优势

| 特性 | 说明 |
|------|------|
| **零卡启动** | Notebook 本身不需要 GPU，训练在云端自动执行 |
| **按需付费** | 仅在训练时占用 GPU 资源 |
| **开箱即用** | 预置主流模型，无需下载权重 |
| **LoRA 微调** | 高效参数微调，几分钟即可完成小规模训练 |

> 🔗 本项目由 [Twinkle](https://github.com/modelscope/twinkle) 框架提供支持 | [GitHub](https://github.com/modelscope/twinkle)

## 第一步：导入依赖并加载环境变量

## 环境安装

首次运行前，请先执行以下安装命令。如已安装可跳过此步。

In [ ]:
!pip install twinkle-kit[tinker] -q

In [ ]:
import numpy as np
import torch
from peft import LoraConfig

from getpass import getpass
from twinkle import get_logger
from twinkle.data_format import Trajectory, Message
from twinkle.preprocessor import Preprocessor
from twinkle.dataset import DatasetMeta
from twinkle_client import init_twinkle_client
from twinkle.dataloader import DataLoader
from twinkle.dataset import LazyDataset
from twinkle_client.model import MultiLoraTransformersModel

logger = get_logger()

api_key = getpass("ModelScope Token: ")

## 第二步：初始化客户端并查询历史训练

In [ ]:
base_model = 'Qwen/Qwen3.6-27B'
base_url = 'http://www.modelscope.cn/twinkle'

client = init_twinkle_client(base_url=base_url, api_key=api_key)

runs = client.list_training_runs()
resume_path = None
for run in runs:
    logger.info(run.model_dump_json(indent=2))
    checkpoints = client.list_checkpoints(run.training_run_id)
    for checkpoint in checkpoints:
        logger.info(checkpoint.model_dump_json(indent=2))
        # resume_path = checkpoint.twinkle_path

print(f'Found {len(runs)} training run(s)')

## 第三步：定义数据预处理器

`LatexOCRProcessor` 将原始数据转为多模态对话格式：
- **user 消息**：包含图片（`<image>` 标记）和指令文本
- **assistant 消息**：图片对应的 LaTeX 公式（训练目标）

数据集中每条样本包含：
- `image`：公式图片
- `text`：对应的 LaTeX 代码

In [ ]:
class LatexOCRProcessor(Preprocessor):

    def __call__(self, rows):
        rows = self.map_col_to_row(rows)
        rows = [self.preprocess(row) for row in rows]
        rows = self.map_row_to_col(rows)
        return rows

    def preprocess(self, row) -> Trajectory:
        return Trajectory(
            messages=[
                Message(
                    role='user',
                    content='<image>Using LaTeX to perform OCR on the image.',
                    images=[row['image']]
                ),
                Message(
                    role='assistant',
                    content=row['text']
                ),
            ]
        )

## 第四步：准备数据集

使用 ModelScope 上的 `AI-ModelScope/LaTeX_OCR` 数据集。

**关键区别**：这里使用 `LazyDataset` 而非 `Dataset`，因为图片数据量较大，延迟加载可以节省内存。

| 参数 | 值 | 说明 |
|------|-----|------|
| 模板 | `Qwen3_5Template` | 支持多模态输入（图片+文本） |
| max_length | 512 | 最大 token 数 |
| data_slice | range(500) | 取前 500 条样本 |

In [ ]:
dataset = LazyDataset(dataset_meta=DatasetMeta('ms://AI-ModelScope/LaTeX_OCR', data_slice=range(500)))

# 使用 Qwen3.5 专用多模态模板
dataset.set_template('Qwen3_5Template', model_id=f'ms://{base_model}', max_length=512)

# 应用 LaTeX OCR 预处理
dataset.map(LatexOCRProcessor)

# 编码
dataset.encode(batched=True)

dataloader = DataLoader(dataset=dataset, batch_size=4)
print(f'Dataset size: {len(dataset)}')

## 第五步：配置模型

与文本训练基本一致，唯一区别是模板使用 `Qwen3_5Template`。

In [ ]:
model = MultiLoraTransformersModel(model_id=f'ms://{base_model}')

lora_config = LoraConfig(target_modules='all-linear')
model.add_adapter_to_model('default', lora_config, gradient_accumulation_steps=2)

model.set_template('Qwen3_5Template')
model.set_processor('InputProcessor', padding_side='right')
model.set_loss('CrossEntropyLoss')
model.set_optimizer('Adam', lr=1e-4)

# 断点续训
if resume_path:
    logger.info(f'Resuming from {resume_path}')
    model.load(resume_path, load_optimizer=True)

logger.info(model.get_train_configs().model_dump())
print('Model configured')

## 第六步：训练循环

与文本训练的训练循环几乎一样，但有一个关键区别：**数据格式转换**。

多模态数据中包含 numpy 数组和 torch 张量（图片特征），在通过网络发送到服务端前需要转为 Python 列表。

```python
# 这段转换逻辑是多模态训练特有的
for sample in batch:
    for key in sample:
        if isinstance(sample[key], np.ndarray):
            sample[key] = sample[key].tolist()
        elif isinstance(sample[key], torch.Tensor):
            sample[key] = sample[key].cpu().numpy().tolist()
```

In [ ]:
for epoch in range(3):
    logger.info(f'Starting epoch {epoch}')
    for step, batch in enumerate(dataloader):
        # 多模态特有：将张量转为列表以适配网络传输
        for sample in batch:
            for key in sample:
                if isinstance(sample[key], np.ndarray):
                    sample[key] = sample[key].tolist()
                elif isinstance(sample[key], torch.Tensor):
                    sample[key] = sample[key].cpu().numpy().tolist()

        # 前向 + 反向
        model.forward_backward(inputs=batch)

        # 梯度裁剪 + 优化器更新
        model.clip_grad_and_step()

        # 每 2 步打印指标
        if step % 2 == 0:
            metric = model.calculate_metric(is_training=True)
            logger.info(f'Current is step {step} of {len(dataloader)}, metric: {metric.result}')

    # 保存检查点
    twinkle_path = model.save(name=f'twinkle-epoch-{epoch}', save_optimizer=True)
    logger.info(f'Saved checkpoint: {twinkle_path}')

## 第七步：上传到 ModelScope Hub（可选）

In [ ]:
from twinkle import init_twinkle_client
from twinkle_client.model import MultiLoraTransformersModel
# 步骤 1：初始化 Twinkle 客户端。
# Tinker 检查点（twinkle:// 路径）由同一检查点服务解析
init_twinkle_client(base_url=base_url, api_key=api_key)

# 步骤 2：创建模型客户端（上传时无需训练状态）
model = MultiLoraTransformersModel(model_id=f'ms://{base_model}')
# ModelScope 目标仓库（必须属于你的账号）
hub_model_id = 'your_username/your-model-name'

# 步骤 3：将检查点上传到 ModelScope Hub。
# 客户端会自动轮询等待上传完成，进度信息会打印到标准输出。
print(f'Uploading {twinkle_path!r} → {hub_model_id!r} ...')
model.upload_to_hub(
    checkpoint_dir=twinkle_path,
    hub_model_id=hub_model_id,
    hub_token=api_key,
)
print(f'Upload complete: https://modelscope.cn/models/{hub_model_id}')

print('Training complete!')

## 推理（Inference）

训练完成后，可以直接使用 **线上服务** 进行推理，无需本地 GPU。

通过 `create_sampling_client` 加载训练好的 LoRA 检查点，即可在线采样生成。

> 将下方 `weight_path` 替换为训练输出的检查点路径（`twinkle://...` 格式）。


In [ ]:
# 推理示例（使用线上服务，无需本地 GPU）
from tinker import types
from twinkle import init_tinker_client, get_logger
from twinkle.data_format import Message, Trajectory
from twinkle.template import Qwen3_5Template

logger = get_logger()

BASE_MODEL = 'Qwen/Qwen3.6-27B'

# TODO: 替换为训练输出的检查点路径
weight_path = '<替换为你的 twinkle:// 检查点路径>'

init_tinker_client()
from tinker import ServiceClient

service_client = ServiceClient(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=api_key,
)

sampling_client = service_client.create_sampling_client(
    model_path=weight_path,
    base_model=BASE_MODEL,
)

# 构造 Prompt（多模态需使用 Qwen3_5Template）
template = Qwen3_5Template(model_id=f'ms://{BASE_MODEL}')
trajectory = Trajectory(
    messages=[
        Message(
            role='user',
            content='<image>Using LaTeX to perform OCR on the image.',
        ),
    ]
)

input_feature = template.encode(trajectory, add_generation_prompt=True)
input_ids = input_feature['input_ids'].tolist()

prompt = types.ModelInput.from_ints(input_ids)
params = types.SamplingParams(max_tokens=256, temperature=0.2)

print('Sampling...')
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=1)
result = future.result()

print('Responses:')
for i, seq in enumerate(result.sequences):
    print(f'{i}: {repr(template.decode(seq.tokens))}')

## 合并 LoRA 权重（可选）

如果需要将 LoRA 权重合并为完整模型（用于无 LoRA 支持的部署场景），可以使用 PEFT 提供的合并功能。

> **注意**：合并操作需要加载完整模型，请在有足够显存的环境下执行。


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_id = 'Qwen/Qwen3.6-27B'
lora_path = '<替换为你的 LoRA 检查点路径>'
output_dir = '<替换为输出目录>'

# 加载基座模型
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype='auto', device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 加载 LoRA 适配器并合并
model = PeftModel.from_pretrained(base_model, lora_path)
merged_model = model.merge_and_unload()

# 保存合并后的完整模型
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'合并完成，模型已保存到 {output_dir}')


合并后的模型为标准 HuggingFace 格式，可直接用 vLLM、Transformers 等框架加载推理：

```bash
# 使用 vLLM 部署合并后的模型
vllm serve <输出目录> --tensor-parallel-size 2
```

> **提示**：对于 Dense 模型（如 Qwen3.6-27B），LoRA 权重可以直接通过 vLLM 的 `enable_lora` 加载，无需合并。只有在不支持动态 LoRA 的部署场景下才需要合并。


## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| 图片加载失败 | 数据集下载不完整 | 重新下载或检查网络 |
| OOM 内存不足 | 图片特征占用显存 | 减小 batch_size 或 max_length |
| 张量序列化错误 | 忘记转换为列表 | 确保训练循环中有 numpy/torch 转 list 逻辑 |
| 模板不匹配 | 使用了通用 Template | 多模态需用 `Qwen3_5Template` |